In [1]:
%useLatestDescriptors
%use dataframe(1.0.0-Beta5n)
%use kandy
%use serialization

In [2]:
@file:Repository("https://repo.gradle.org/gradle/libs-releases") //
@file:DependsOn("org.gradle:gradle-tooling-api:9.6.0")

import org.gradle.tooling.GradleConnector
import java.io.File

GradleConnector.newConnector().forProjectDirectory(File("./../../")).connect().use { connection ->
    connection.newBuild()
            .forTasks(
                "clean",
                "crcBenchmark",
                "deflateBenchmark",
                "inflateBenchmark"
            )
            .setStandardOutput(System.out)
            .setStandardError(System.err)
            .run()
}

Calculating task graph as configuration cache cannot be reused because file 'settings.gradle.kts' has changed.
Type-safe project accessors is an incubating feature.
Mocha test framework for Wasm target is not supported. For KotlinWasmNode used
Mocha test framework for Wasm target is not supported. For KotlinWasmNode used
> Task :kompress-benchmarks:cleanWasmJsNodeTest UP-TO-DATE
> Task :kompress-benchmarks:cleanMacosArm64Test UP-TO-DATE
> Task :kompress-benchmarks:cleanLinuxX64Test UP-TO-DATE
> Task :kompress-benchmarks:cleanWasmJsBrowserTest UP-TO-DATE
> Task :kompress-benchmarks:cleanJvmTest UP-TO-DATE
> Task :kompress-benchmarks:cleanWatchosSimulatorArm64Test UP-TO-DATE
> Task :kompress-benchmarks:cleanMingwX64Test UP-TO-DATE
> Task :kompress-benchmarks:cleanJsNodeTest UP-TO-DATE
> Task :kompress-benchmarks:cleanTvosSimulatorArm64Test UP-TO-DATE
> Task :kompress-benchmarks:cleanJsBrowserTest UP-TO-DATE
> Task :kompress-benchmarks:cleanIosSimulatorArm64Test UP-TO-DATE
> Task :kompres

In [5]:
@file:OptIn(ExperimentalSerializationApi::class)

import kotlinx.serialization.ExperimentalSerializationApi
import kotlinx.serialization.SerialName
import kotlinx.serialization.Serializable
import kotlinx.serialization.json.Json
import kotlinx.serialization.json.decodeFromStream
import org.jetbrains.kotlinx.kandy.dsl.plot
import org.jetbrains.kotlinx.kandy.letsplot.layers.bars
import java.nio.file.Path
import kotlin.io.path.PathWalkOption
import kotlin.io.path.extension
import kotlin.io.path.inputStream
import kotlin.io.path.name
import kotlin.io.path.nameWithoutExtension
import kotlin.io.path.walk

fun String.suffixSimilarity(other: String): Double {
    val a = this.reversed()
    val b = other.reversed()
    val maxLen = maxOf(a.length, b.length)
    if (maxLen == 0) return 1.0
    var match = 0
    for (i in 0 until minOf(a.length, b.length)) {
        if (a[i] == b[i]) match++ else break
    }
    return match.toDouble() / maxLen
}

inline fun <T> List<T>.groupBySimilarityChain(crossinline selector: (T) -> String): List<T> {
    if (isEmpty()) return emptyList()
    val unused = toMutableSet()
    val result = ArrayList<T>()
    var current = unused.first()
    result += current
    unused -= current
    while (unused.isNotEmpty()) {
        val next = unused.maxBy { selector(it).suffixSimilarity(selector(current)) }
        result += next
        unused -= next
        current = next
    }
    return result
}

@Serializable
data class PrimaryMetric(
    val score: Double,
    val scoreError: Double,
    val scoreUnit: String
)

@Serializable
data class Benchmark(
    @SerialName("benchmark") val name: String,
    val warmupIterations: Int,
    val measurementIterations: Int,
    val primaryMetric: PrimaryMetric
) {
    inline val cleanName: String
        get() = name.substringBeforeLast('.').substringAfterLast('.')
}

val json: Json = Json { ignoreUnknownKeys = true }

Path.of("./../build/reports/benchmarks")
        .walk()
        .filter { filePath -> filePath.extension == "json" }
        .map { filePath ->
            filePath.inputStream().use { stream ->
                filePath to json.decodeFromStream<List<Benchmark>>(stream)
            }
        }.map { (filePath, report) ->
            val sortedReport = report.groupBySimilarityChain(Benchmark::name)
            val plotName = filePath.parent.parent.name
            val platform = filePath.nameWithoutExtension
            val df = dataFrameOf(
                "name" to sortedReport.map(Benchmark::cleanName),
                "score" to sortedReport.map { benchmark -> benchmark.primaryMetric.score },
                "score_min" to sortedReport.map { benchmark -> benchmark.primaryMetric.score - benchmark.primaryMetric.scoreError },
                "score_max" to sortedReport.map { benchmark -> benchmark.primaryMetric.score + benchmark.primaryMetric.scoreError }
            )
            val plot = plot(df) {
                layout {
                    size = 1200 to 800
                    title = "$plotName ($platform)" // Parent of parent is suite name
                    xAxisLabel = "Implementation"
                    yAxisLabel = "MiB/s"
                    theme = Theme.DARCULA
                }
                bars {
                    x("name")
                    y("score") {
                        axis {
                            breaks(format = ".2f")
                        }
                    }
                    fillColor("name")
                }
                errorBars {
                    x("name")
                    yMin("score_min")
                    yMax("score_max")
                }
            }
            plot.save("./../../docs/${plotName}_$platform.png")
            plot
        }.forEach(::DISPLAY)

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="3Z6nis" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("3Z6nis");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"crc (jvm)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[1370.6904879563378,13666.014493048022],
"name":["CRC32Benchmark","JvmCRC32Benchmark"],
"score_max":[1400.496588329139,13959.688411020908],
"score_min":[1340.8843875835366,13372.340575075135]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"111"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CRC32Benchmark 
 
 
 
 
 
 
 
 
 JvmCRC32Benchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 1000.00 
 
 
 
 
 
 
 2000.00 
 
 
 
 
 
 
 3000.00 
 
 
 
 
 
 
 4000.00 
 
 
 
 
 
 
 5000.00 
 
 
 
 
 
 
 6000.00 
 
 
 
 
 
 
 7000.00 
 
 
 
 
 
 
 8000.00 
 
 
 
 
 
 
 9000.00 
 
 
 
 
 
 
 10000.00 
 
 
 
 
 
 
 11000.00 
 
 
 
 
 
 
 12000.00 
 
 
 
 
 
 
 13000.00 
 
 
 
 
 
 
 14000.00 
 
 
 
 
 
 
 
 
 crc (jvm) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CRC32Benchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 JvmCRC32Benchmark

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="9SvMsc" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("9SvMsc");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"crc (js)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[497.62846894663363],
"name":["CRC32Benchmark"],
"score_max":[534.023461481747],
"score_min":[461.23347641152037]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"115"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CRC32Benchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 50.00 
 
 
 
 
 
 
 100.00 
 
 
 
 
 
 
 150.00 
 
 
 
 
 
 
 200.00 
 
 
 
 
 
 
 250.00 
 
 
 
 
 
 
 300.00 
 
 
 
 
 
 
 350.00 
 
 
 
 
 
 
 400.00 
 
 
 
 
 
 
 450.00 
 
 
 
 
 
 
 500.00 
 
 
 
 
 
 
 550.00 
 
 
 
 
 
 
 
 
 crc (js) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CRC32Benchmark

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="WQ9M9x" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("WQ9M9x");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"crc (wasmWasi)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[946.4888082981652],
"name":["CRC32Benchmark"],
"score_max":[952.6244920419508],
"score_min":[940.3531245543795]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"119"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CRC32Benchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 50.00 
 
 
 
 
 
 
 100.00 
 
 
 
 
 
 
 150.00 
 
 
 
 
 
 
 200.00 
 
 
 
 
 
 
 250.00 
 
 
 
 
 
 
 300.00 
 
 
 
 
 
 
 350.00 
 
 
 
 
 
 
 400.00 
 
 
 
 
 
 
 450.00 
 
 
 
 
 
 
 500.00 
 
 
 
 
 
 
 550.00 
 
 
 
 
 
 
 600.00 
 
 
 
 
 
 
 650.00 
 
 
 
 
 
 
 700.00 
 
 
 
 
 
 
 750.00 
 
 
 
 
 
 
 800.00 
 
 
 
 
 
 
 850.00 
 
 
 
 
 
 
 900.00 
 
 
 
 
 
 
 950.00 
 
 
 
 
 
 
 1000.00 
 
 
 
 
 
 
 
 
 crc (wasmWasi) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CRC32Benchmark

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="WrQM1O" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("WrQM1O");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"crc (wasmJs)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[932.2249685635185],
"name":["CRC32Benchmark"],
"score_max":[943.892075741184],
"score_min":[920.5578613858529]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"123"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CRC32Benchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 50.00 
 
 
 
 
 
 
 100.00 
 
 
 
 
 
 
 150.00 
 
 
 
 
 
 
 200.00 
 
 
 
 
 
 
 250.00 
 
 
 
 
 
 
 300.00 
 
 
 
 
 
 
 350.00 
 
 
 
 
 
 
 400.00 
 
 
 
 
 
 
 450.00 
 
 
 
 
 
 
 500.00 
 
 
 
 
 
 
 550.00 
 
 
 
 
 
 
 600.00 
 
 
 
 
 
 
 650.00 
 
 
 
 
 
 
 700.00 
 
 
 
 
 
 
 750.00 
 
 
 
 
 
 
 800.00 
 
 
 
 
 
 
 850.00 
 
 
 
 
 
 
 900.00 
 
 
 
 
 
 
 950.00 
 
 
 
 
 
 
 
 
 crc (wasmJs) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CRC32Benchmark

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="nAgzVT" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("nAgzVT");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"crc (linuxX64)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[1355.2651807077623],
"name":["CRC32Benchmark"],
"score_max":[1378.545644736478],
"score_min":[1331.9847166790466]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"127"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CRC32Benchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 100.00 
 
 
 
 
 
 
 200.00 
 
 
 
 
 
 
 300.00 
 
 
 
 
 
 
 400.00 
 
 
 
 
 
 
 500.00 
 
 
 
 
 
 
 600.00 
 
 
 
 
 
 
 700.00 
 
 
 
 
 
 
 800.00 
 
 
 
 
 
 
 900.00 
 
 
 
 
 
 
 1000.00 
 
 
 
 
 
 
 1100.00 
 
 
 
 
 
 
 1200.00 
 
 
 
 
 
 
 1300.00 
 
 
 
 
 
 
 1400.00 
 
 
 
 
 
 
 
 
 crc (linuxX64) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 CRC32Benchmark

...

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="qRVQoH" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("qRVQoH");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"inflate (jvm)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[1454.4946663910318,436.33698754938604,1556.130908543423,430.3185404571448,1614.5344529151726,432.44792416936536],
"name":["InflaterDefaultLevelBenchmark","JvmInflaterDefaultLevelBenchmark","InflaterMaxLevelBenchmark","JvmInflaterMaxLevelBenchmark","InflaterMinLevelBenchmark","JvmInflaterMinLevelBenchmark"],
"score_max":[1718.989268183965,441.7123882618153,1617.3529086309238,438.48020612407134,1691.9778998394827,440.27297357366064],
"score_min":[1190.0000645980988,430.96158683695677,1494.908908455922,422.1568747902183,1537.0910059908624,424.6228747650701]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"147"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 JvmInflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 InflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 JvmInflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 InflaterMinLevelBenchmark 
 
 
 
 
 
 
 
 
 JvmInflaterMinLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 200.00 
 
 
 
 
 
 
 400.00 
 
 
 
 
 
 
 600.00 
 
 
 
 
 
 
 800.00 
 
 
 
 
 
 
 1000.00 
 
 
 
 
 
 
 1200.00 
 
 
 
 
 
 

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="XtfcXT" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("XtfcXT");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"inflate (js)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[168.2796638150536,194.99653165191538,96.68666756191905],
"name":["InflaterDefaultLevelBenchmark","InflaterMaxLevelBenchmark","InflaterMinLevelBenchmark"],
"score_max":[179.69814677538614,197.6609121058207,97.91783361149865],
"score_min":[156.86118085472106,192.33215119801005,95.45550151233945]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"151"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 InflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 InflaterMinLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 10.00 
 
 
 
 
 
 
 20.00 
 
 
 
 
 
 
 30.00 
 
 
 
 
 
 
 40.00 
 
 
 
 
 
 
 50.00 
 
 
 
 
 
 
 60.00 
 
 
 
 
 
 
 70.00 
 
 
 
 
 
 
 80.00 
 
 
 
 
 
 
 90.00 
 
 
 
 
 
 
 100.00 
 
 
 
 
 
 
 110.00 
 
 
 
 
 
 
 120.00 
 
 
 
 
 
 
 130.00 
 
 
 
 
 
 
 140.00 
 
 
 
 
 
 
 150.00 
 
 
 
 
 
 
 160.00 
 
 
 
 
 
 
 170.00 
 
 
 
 
 
 
 180.00 
 
 
 
 
 
 
 190.00 
 
 
 
 
 
 
 200.00 
 
 
 
 
 
 
 
 
 inflate (js) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="2HITVK" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("2HITVK");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"inflate (wasmWasi)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[453.4894905629443,446.126313039679,445.6744014558082],
"name":["InflaterDefaultLevelBenchmark","InflaterMaxLevelBenchmark","InflaterMinLevelBenchmark"],
"score_max":[468.2632965448568,454.7138289594187,453.89725659737786],
"score_min":[438.7156845810318,437.5387971199393,437.45154631423856]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"155"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 InflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 InflaterMinLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 50.00 
 
 
 
 
 
 
 100.00 
 
 
 
 
 
 
 150.00 
 
 
 
 
 
 
 200.00 
 
 
 
 
 
 
 250.00 
 
 
 
 
 
 
 300.00 
 
 
 
 
 
 
 350.00 
 
 
 
 
 
 
 400.00 
 
 
 
 
 
 
 450.00 
 
 
 
 
 
 
 
 
 inflate (wasmWasi) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterMinLevelBenchmark

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="iy7rDl" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("iy7rDl");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"inflate (wasmJs)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[455.5291078074677,435.89468792125007,439.2636931435442],
"name":["InflaterDefaultLevelBenchmark","InflaterMaxLevelBenchmark","InflaterMinLevelBenchmark"],
"score_max":[464.5918103106804,444.9005685098091,449.24117783551486],
"score_min":[446.46640530425503,426.88880733269104,429.2862084515735]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"159"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 InflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 InflaterMinLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 50.00 
 
 
 
 
 
 
 100.00 
 
 
 
 
 
 
 150.00 
 
 
 
 
 
 
 200.00 
 
 
 
 
 
 
 250.00 
 
 
 
 
 
 
 300.00 
 
 
 
 
 
 
 350.00 
 
 
 
 
 
 
 400.00 
 
 
 
 
 
 
 450.00 
 
 
 
 
 
 
 
 
 inflate (wasmJs) 
 
 
 
 
 MiB/s 
 
 
 
 
 Implementation 
 
 
 
 
 
 
 
 
 name 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterMinLevelBenchmark

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="e2Sko0" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 1200.0, 
 height: 800.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("e2Sko0");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"inflate (linuxX64)"
},
"mapping":{
},
"guides":{
"x":{
"title":"Implementation"
},
"y":{
"title":"MiB/s"
}
},
"data":{
"score":[516.1177788303593,340.2597918263609,582.4694535178454,321.80271636156465,541.7677562899344,326.6974608877934],
"name":["InflaterDefaultLevelBenchmark","NativeInflaterDefaultLevelBenchmark","InflaterMaxLevelBenchmark","NativeInflaterMaxLevelBenchmark","InflaterMinLevelBenchmark","NativeInflaterMinLevelBenchmark"],
"score_max":[540.0464578894605,343.2179118849623,592.6250511757131,333.36626312866804,552.2467253750991,333.26060777311704],
"score_min":[492.18909977125804,337.30167176775956,572.3138558599777,310.23916959446126,531.2887872047696,320.13431400246975]
},
"ggsize":{
"width":1200.0,
"height":800.0
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"format":".2f",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
},{
"aesthetic":"x",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"name",
"y":"score",
"fill":"name"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data":{
}
},{
"mapping":{
"x":"name",
"ymin":"score_min",
"ymax":"score_max"
},
"stat":"identity",
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"errorbar",
"data":{
}
}],
"theme":{
"flavor":"darcula"
},
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"name"
},{
"type":"float",
"column":"score"
},{
"type":"float",
"column":"score_min"
},{
"type":"float",
"column":"score_max"
}]
},
"spec_id":"163"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 InflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 NativeInflaterDefaultLevelBenchmark 
 
 
 
 
 
 
 
 
 InflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 NativeInflaterMaxLevelBenchmark 
 
 
 
 
 
 
 
 
 InflaterMinLevelBenchmark 
 
 
 
 
 
 
 
 
 NativeInflaterMinLevelBenchmark 
 
 
 
 
 
 
 
 
 
 
 0.00 
 
 
 
 
 
 
 50.00 
 
 
 
 
 
 
 100.00 
 
 
 
 
 
 
 150.00 
 
 
 
 
 
 
 200.00 
 
 
 
 
 
 
 250.00 
 
 
